# Detekcja AI-Generated Images — Inferencja

Notatnik wczytuje zapisany model i wykonuje predykcję na zdjęciu wybranym z dysku.

### Wymagania
- Plik modelu `saved_models/best_model.keras` (lub `final_model.keras`) z notatnika treningowego
- Biblioteki: `tensorflow`, `pillow`, `matplotlib`


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ─── Konfiguracja ───
IMG_SIZE = 224

# Ścieżka do modelu — zmień jeśli zapisałeś w innym miejscu
MODEL_PATH = "saved_models/best_model.keras"
# MODEL_PATH = "saved_models/final_model.keras"         # alternatywa
# MODEL_PATH = "saved_models/best_model_finetuned.keras" # po fine-tuningu

LABEL_A_MAP = {0: "Real", 1: "AI-Generated"}
LABEL_B_MAP = {
    0: "Real",
    1: "Stable Diffusion 2.1",
    2: "Stable Diffusion XL",
    3: "Stable Diffusion 3",
    4: "DALL-E 3",
    5: "Midjourney 6"
}

In [ ]:
# ─── Wczytanie modelu ───
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Nie znaleziono modelu: {MODEL_PATH}\n"
        f"Dostępne pliki w saved_models/: {os.listdir('saved_models') if os.path.isdir('saved_models') else 'brak folderu'}"
    )

model = tf.keras.models.load_model(MODEL_PATH)
print(f"✅ Model wczytany z: {MODEL_PATH}")
print(f"   Input shape:  {model.input_shape}")
print(f"   Output shape: {model.output_shape}")
print(f"   Parametry:    {model.count_params():,}")

In [ ]:
def preprocess(image_pil):
    """PIL Image → float32 tensor [0, 255] (EfficientNet normalizuje wewnętrznie)."""
    img = image_pil.convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img, dtype=np.float32)  # [0, 255] — BEZ dzielenia!
    return img_array

def predict_image(model, image_pil, threshold=0.5):
    """Wykonuje predykcję i zwraca wynik."""
    img = preprocess(image_pil)
    batch = np.expand_dims(img, axis=0)  # (1, 224, 224, 3)

    prob = model.predict(batch, verbose=0)[0][0]

    result = {
        'probability_ai': float(prob),
        'probability_real': float(1 - prob),
        'label': 'AI-Generated' if prob > threshold else 'Real',
        'confidence': float(max(prob, 1 - prob)) * 100,
        'threshold': threshold,
    }
    return result

## Wybierz zdjęcie z dysku
Uruchom poniższą komórkę — pojawi się przycisk **Upload**. Wybierz jedno lub kilka zdjęć.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Widget do uploadu plików
uploader = widgets.FileUpload(
    accept='image/*',     # tylko obrazy
    multiple=True,        # można wybrać kilka
    description='Wybierz zdjęcia'
)

display(HTML("<h3>📁 Kliknij przycisk poniżej aby wybrać zdjęcia z laptopa:</h3>"))
display(uploader)

In [ ]:
# ─── Predykcja na przesłanych zdjęciach ───
# Uruchom tę komórkę PO wybraniu zdjęć w widgetcie powyżej

import io

if not uploader.value:
    print("⚠️  Najpierw wybierz zdjęcia w widgetcie powyżej, potem uruchom tę komórkę ponownie.")
else:
    uploaded_files = uploader.value
    n = len(uploaded_files)

    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 7 * rows))

    if n == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    for i, file_info in enumerate(uploaded_files):
        # Odczytaj obraz z uploadera
        img_bytes = file_info['content']
        img_pil = Image.open(io.BytesIO(img_bytes))
        filename = file_info['name']

        # Predykcja
        result = predict_image(model, img_pil)

        # Wizualizacja
        ax = axes[i]
        ax.imshow(img_pil)

        label = result['label']
        conf = result['confidence']
        p_ai = result['probability_ai']

        color = '#e74c3c' if label == 'AI-Generated' else '#2ecc71'

        ax.set_title(
            f"{label}  ({conf:.1f}%)",
            fontsize=14, fontweight='bold', color=color,
            pad=10
        )
        ax.set_xlabel(
            f"{filename}\n"
            f"P(AI) = {p_ai:.4f}  |  P(Real) = {1-p_ai:.4f}",
            fontsize=10
        )
        ax.set_xticks([]); ax.set_yticks([])

        # Ramka w kolorze predykcji
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    # Ukryj puste subploty
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Wyniki detekcji AI", fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Podsumowanie tabelaryczne
    print("\n" + "=" * 65)
    print(f"{'Plik':<30s} {'Wynik':<16s} {'P(AI)':<10s} {'Pewność'}")
    print("=" * 65)
    for file_info in uploaded_files:
        img_pil = Image.open(io.BytesIO(file_info['content']))
        r = predict_image(model, img_pil)
        icon = "🤖" if r['label'] == 'AI-Generated' else "📷"
        print(f"{file_info['name']:<30s} {icon} {r['label']:<13s} {r['probability_ai']:<10.4f} {r['confidence']:.1f}%")
    print("=" * 65)

## Alternatywa — podaj ścieżkę do pliku
Jeśli widget nie działa (np. w terminalu), możesz podać ścieżkę bezpośrednio:


In [ ]:
# ─── Predykcja na zdjęciu z podanej ścieżki ───
# Zmień ścieżkę na swoją:
IMAGE_PATH = "test_image.jpg"

if os.path.exists(IMAGE_PATH):
    img_pil = Image.open(IMAGE_PATH)
    result = predict_image(model, img_pil)

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img_pil)

    label = result['label']
    conf = result['confidence']
    color = '#e74c3c' if label == 'AI-Generated' else '#2ecc71'

    ax.set_title(f"{label}  ({conf:.1f}%)", fontsize=16, fontweight='bold', color=color)
    ax.set_xlabel(
        f"P(AI) = {result['probability_ai']:.4f}  |  P(Real) = {result['probability_real']:.4f}",
        fontsize=12
    )
    ax.set_xticks([]); ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

    plt.tight_layout()
    plt.show()
else:
    print(f"Plik nie istnieje: {IMAGE_PATH}")
    print(f"Zmień zmienną IMAGE_PATH na prawidłową ścieżkę.")

## Predykcja na całym folderze

In [ ]:
# ─── Predykcja na wszystkich obrazach w folderze ───
FOLDER_PATH = "test_images"   # zmień na swój folder
SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}

if os.path.isdir(FOLDER_PATH):
    image_files = sorted([
        f for f in os.listdir(FOLDER_PATH)
        if os.path.splitext(f)[1].lower() in SUPPORTED_EXT
    ])

    if not image_files:
        print(f"Brak obrazów w {FOLDER_PATH}")
    else:
        print(f"Znaleziono {len(image_files)} obrazów w {FOLDER_PATH}\n")

        results = []
        for fname in image_files:
            img_pil = Image.open(os.path.join(FOLDER_PATH, fname))
            r = predict_image(model, img_pil)
            r['filename'] = fname
            results.append(r)

        # Podsumowanie
        n_ai = sum(1 for r in results if r['label'] == 'AI-Generated')
        n_real = len(results) - n_ai

        print(f"{'Plik':<35s} {'Wynik':<16s} {'P(AI)':<10s} {'Pewność'}")
        print("-" * 70)
        for r in results:
            icon = "🤖" if r['label'] == 'AI-Generated' else "📷"
            print(f"{r['filename']:<35s} {icon} {r['label']:<13s} {r['probability_ai']:<10.4f} {r['confidence']:.1f}%")
        print("-" * 70)
        print(f"\nPodsumowanie: 📷 Real: {n_real} | 🤖 AI: {n_ai} | Razem: {len(results)}")

        # Wykres rozkładu prawdopodobieństw
        probs = [r['probability_ai'] for r in results]
        fig, ax = plt.subplots(figsize=(10, 4))
        colors = ['#e74c3c' if p > 0.5 else '#2ecc71' for p in probs]
        bars = ax.bar(range(len(probs)), probs, color=colors)
        ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='Próg decyzji (0.5)')
        ax.set_xticks(range(len(probs)))
        ax.set_xticklabels([r['filename'][:15] for r in results], rotation=45, ha='right', fontsize=8)
        ax.set_ylabel("P(AI-Generated)")
        ax.set_title("Prawdopodobieństwo AI dla każdego obrazu", fontweight='bold')
        ax.legend()
        ax.set_ylim(0, 1)
        plt.tight_layout()
        plt.show()
else:
    print(f"Folder nie istnieje: {FOLDER_PATH}")
    print("Utwórz folder i umieść w nim zdjęcia do analizy.")

## Interpretacja wyników

| Pewność | Interpretacja |
|---------|---------------|
| **> 90%** | Model jest bardzo pewny — wynik wiarygodny |
| **70–90%** | Dość pewny — warto zweryfikować |
| **50–70%** | Niepewny — wynik na granicy, nie polegaj na nim |
| **< 50%** | Model nie potrafi zdecydować |

**Pamiętaj:** model trenowany był na konkretnych generatorach (SD 2.1, SDXL, SD3, DALL-E 3, Midjourney 6).
Obrazy z innych modeli (np. FLUX, Imagen) mogą dawać nieprzewidywalne wyniki.
